<a href="https://colab.research.google.com/github/samilnamli/transformer_experiments/blob/runpod/voxpopuli-main-results/notebooks/colab/main_results_voxpopuli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VoxPopuli — main-results comparison table (Colab)

Paper-ready comparison table for **HIT-ASR** vs. all baselines on VoxPopuli,
run as a **many-seed repeated-random-split cross-validation**.

**What runs** (config: `experiment=main_results_voxpopuli_colab`):

| group | methods |
|---|---|
| training-free baselines | single-hubert / -whisper / -wav2vec2, random, weighted-random, oracle, ROVER, weighted-ROVER |
| trainable rival | **mlp_pool** — trained with the *same* composite loss as HIT-ASR (weighted-WER + hard-CE + soft-CE), **not** hard-CE-only |
| our method | **hierarchical_transformer** (HIT-ASR) |

**Cross-validation.** Each seed re-derives *both* the 80/10/10 train/val/test
split *and* the model init (`fixed_data_split=false`). Splits are independent
random resamples of the same dataset — repeated-random-split (Monte-Carlo) CV,
which is exactly the precondition for the Nadeau–Bengio corrected paired
t-test that runs at the end. The seed list is 20 fun nerd numbers.

**Aggregation is automatic.** `BaseExperiment.run()` aggregates across seeds
(`wer_mean ± wer_sem`), logs the comparison table, and runs the pairwise
significance tests in the same process — there is no separate aggregation
step to launch. The final cell simply reads those logged tables back and
renders them inline.

> ⏱️ **Runtime.** 2 trainable models × 20 seeds × ≤50 epochs is many GPU-hours.
> The run cell has a `QUICK_SEEDS` knob for a fast 3-seed pass. Use an A100/L4
> high-RAM runtime; a Whisper re-decode step (~1 h) runs once up front.

## 0. GPU check

In [ ]:
!nvidia-smi

## 1. Secrets & git

Add these under Colab **🔑 Secrets** (left sidebar). Only `HF_TOKEN` is
strictly required (to fetch data). The `DAGSHUB_*` / `WANDB_*` secrets are
optional — without them, MLflow logs to a **local `./mlruns`** folder and the
final cell still works.

In [ ]:
import os
try:
    from google.colab import userdata
    def _get(k):
        try:
            return userdata.get(k)
        except Exception:
            return None
except Exception:
    def _get(k):
        return os.environ.get(k)

# Required: HuggingFace token (dataset download).
if _get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = _get('HF_TOKEN')

# Optional: remote MLflow tracking on DagsHub. Skip -> local ./mlruns.
if _get('DAGSHUB_USER_TOKEN'):
    os.environ['DAGSHUB_USER_TOKEN'] = _get('DAGSHUB_USER_TOKEN')
    os.environ['DAGSHUB_TRACKING_URI'] = (
        _get('DAGSHUB_TRACKING_URI')
        or 'https://dagshub.com/huseyin-karaca/s2t-tr-dev.mlflow')
    os.environ['DAGSHUB_USERNAME'] = _get('DAGSHUB_USERNAME') or 'huseyin-karaca'
    print('MLflow -> DagsHub:', os.environ['DAGSHUB_TRACKING_URI'])
else:
    print('MLflow -> local ./mlruns (no DAGSHUB_USER_TOKEN set)')

# Optional: Weights & Biases.
if _get('WANDB_API_KEY'):
    os.environ['WANDB_API_KEY'] = _get('WANDB_API_KEY')

!git config --global user.name  'huseyin-karaca'
!git config --global user.email 'huseyinkaraccca@gmail.com'

## 2. Clone repo & install deps

Installs `uv`, creates a Python 3.10 venv, and `uv sync`. On Linux the
project pins `torch==2.6.0+cu124` (Samil's RunPod fix), which is CUDA-compatible
with Colab GPUs too.

In [ ]:
%cd /content
!pip -q install uv
![ -d transformer_experiments ] || git clone https://github.com/samilnamli/transformer_experiments.git
%cd /content/transformer_experiments
!git checkout runpod/voxpopuli-main-results && git pull --ff-only
!uv venv --python 3.10
!uv sync

## 3. Download VoxPopuli parquet

Fetches the cached frame-level encoder features (~24 GB) to
`configs/data/processed/facebook_voxpopuli/`.

In [ ]:
%cd /content/transformer_experiments
!uv run make download_voxpopuli

## 4. Whisper re-decode (required, run once)

Builds the corrected Whisper WER/transcription overrides `.npz` that the
VoxPopuli data config points at (fixed language + anti-repetition + the GPU
fp16 dtype fix). **The run will `FileNotFoundError` without this file.**
Takes roughly an hour on a Colab GPU. The sanity check confirms shapes line up.

In [ ]:
%cd /content/transformer_experiments
!uv run python scripts/redecode_voxpopuli_whisper_overrides_from_features.py \
    --input-parquet configs/data/processed/facebook_voxpopuli/combined_features_with_transcripts.parquet \
    --output-npz data/processed/facebook_voxpopuli/whisper_decode_overrides_from_features.npz
!uv run python scripts/check_voxpopuli.py

## 5. Run the comparison table (all seeds)

`QUICK_SEEDS=None` uses the full 20-seed list from the config. Set it to a
short list (e.g. `[42, 1337, 73]`) for a fast pass. Bump `BATCH_SIZE` to 192
on an A100-40GB. Aggregation + significance tests run automatically at the end.

In [ ]:
# --- knobs ---
QUICK_SEEDS = None      # None = all 20 config seeds; e.g. [42, 1337, 73] for a fast run
BATCH_SIZE  = 128       # 128 fits L4/T4-high-RAM; 192 on A100-40GB
EAGER_LOAD  = False     # True is faster across seeds but needs a high-RAM runtime

ov = f'data.batch_size={BATCH_SIZE} data.eager_load={str(EAGER_LOAD).lower()}'
if QUICK_SEEDS:
    ov += " experiment.seeds='%s'" % str(QUICK_SEEDS).replace(' ', '')

cmd = ('cd /content/transformer_experiments && '
       'uv run python run.py experiment=main_results_voxpopuli_colab ' + ov)
print(cmd)
!{cmd}

## 6. Aggregate metrics & pairwise significance

Reads the tables logged by the run back from MLflow and renders them: the
across-seed `wer_mean ± wer_sem` comparison and the Nadeau–Bengio corrected
paired t-tests (with Holm correction). Works against DagsHub or local
`./mlruns`, whichever the run used.

In [ ]:
import os, mlflow, pandas as pd
pd.set_option('display.max_rows', None); pd.set_option('display.width', 200)

uri = (os.environ.get('DAGSHUB_TRACKING_URI')
       or os.environ.get('MLFLOW_TRACKING_URI') or 'mlruns')
mlflow.set_tracking_uri(uri)
EXPERIMENT = 'main_results_voxpopuli_colab'   # mlflow experiment = hydra choice name
PARENT_RUN = 'colab_main'
print('tracking_uri =', uri, '| experiment =', EXPERIMENT)

exp = mlflow.get_experiment_by_name(EXPERIMENT)
assert exp is not None, f'experiment {EXPERIMENT!r} not found at {uri}'
runs = mlflow.search_runs(
    [exp.experiment_id],
    filter_string=f"tags.`mlflow.runName` = '{PARENT_RUN}'",
    order_by=['start_time DESC'], max_results=1)
assert len(runs), f'no parent run named {PARENT_RUN!r} yet'
run_id = runs.iloc[0]['run_id']
print('parent run_id =', run_id)

def show(artifact, title):
    try:
        df = mlflow.load_table(artifact_file=artifact, run_ids=[run_id])
        print('\n==', title, '==')
        display(df)
        return df
    except Exception as e:
        print(f'[{title}] not available yet: {e}')
        return None

comp = show('results/test_wer_comparison.json', 'Test WER — across-seed mean ± SEM')
stat = show('results/pairwise_stat_tests.json', 'Pairwise Nadeau-Bengio corrected t-tests')